# MVP Recruitment Workflow Demo

This notebook walks through the public-data recruitment decision-support MVP. It uses the sample CSV included in this package. Replace `data/sample_players.csv` with real player data following `docs/data_schema.md`.

## 1. Load the pipeline

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.scoring import (
    load_config, prepare_players, add_role_performance, add_reliability,
    add_value_model, squad_need_board, build_longlist, rank_shortlist, qa_flags
)

config = load_config(ROOT / 'configs' / 'celtic_style_scenario.yaml')
df = pd.read_csv(ROOT / 'data' / 'sample_players.csv')
df.head()

## 2. Prepare players and calculate role performance

Role scores are calculated within role family using robust standardisation. Low-minute outputs are shrunk towards the peer average to reduce per-90 sample-size bias.

In [ ]:
players = prepare_players(df)
players = add_role_performance(players)
players = add_reliability(players)
players, model_info = add_value_model(players)
model_info

## 3. Squad need hypothesis

This is not an internal club recommendation. It is a public-data hypothesis based on depth, reliable depth, minutes dependency, age risk, performance gap and availability flags.

In [ ]:
roles = list(config['priority_roles'].keys())
need_board = squad_need_board(players, config['club']['name'], roles)
need_board

## 4. Longlist

The longlist applies transparent hard filters from the YAML scenario.

In [ ]:
longlist = build_longlist(players, config)
longlist[['player','team','league','position','role_family','age','minutes','market_value_eur']].head(10)

## 5. Shortlist ranking

The final score is configurable by role and combines role performance, squad complementarity, value intelligence, reliability and league translation risk.

In [ ]:
ranked = rank_shortlist(longlist, players, config)
shortlist = ranked.head(config['outputs']['final_shortlist_n']).copy()
shortlist['qa_flags'] = shortlist.apply(qa_flags, axis=1)
shortlist[['player','team','league','role_family','age','minutes','market_value_eur','final_score','qa_flags']]

## 6. Export outputs

In [ ]:
out = ROOT / 'outputs'
out.mkdir(exist_ok=True)
need_board.to_csv(out / 'need_board_from_notebook.csv', index=False)
longlist.to_csv(out / 'longlist_from_notebook.csv', index=False)
ranked.to_csv(out / 'ranked_candidates_from_notebook.csv', index=False)
shortlist.to_csv(out / 'shortlist_from_notebook.csv', index=False)
print(f'Wrote outputs to {out}')

## Next steps

1. Replace the synthetic sample with a real player dataset.
2. Add a historical backtest.
3. Add sensitivity analysis across alternative weights.
4. Add event-data validation using StatsBomb Open Data and socceraction.
5. Add video validation notes for the top candidates.